In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
project_catalog='data_project'
spark.sql(f"USE CATALOG {project_catalog}")

In [0]:
%sql
-- 1. Region Dimension
CREATE TABLE IF NOT EXISTS gold.dim_region (
    postal_code STRING NOT NULL,
    province STRING,
    city_regency STRING,
    district STRING,
    subdistrict_village STRING,
    CONSTRAINT pk_dim_region PRIMARY KEY (postal_code) RELY
);

-- 2. Location Dimension
CREATE TABLE IF NOT EXISTS gold.dim_location (
    location_id STRING NOT NULL,
    location_name STRING,
    location_tags STRING,
    address STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    location_url STRING,
    website STRING,
    phone_number STRING,
    operating_hours STRING,
    updated_at TIMESTAMP,
    CONSTRAINT pk_dim_location PRIMARY KEY (location_id) RELY
);

-- 3. Location Metrics Fact (Using scraped_date DATE directly)
CREATE TABLE IF NOT EXISTS gold.fact_location_metrics (
    location_id STRING NOT NULL,
    postal_code STRING NOT NULL,
    rating DOUBLE,
    reviews INT,
    scraped_at TIMESTAMP,
    cleaned_at TIMESTAMP,
    CONSTRAINT fk_fact_location FOREIGN KEY (location_id) REFERENCES gold.dim_location(location_id),
    CONSTRAINT fk_fact_region FOREIGN KEY (postal_code) REFERENCES gold.dim_region(postal_code)
);

In [0]:
%sql
MERGE INTO gold.dim_region AS t
USING (
    SELECT 
        COALESCE(NULLIF(TRIM(postal_code), ''), 'UNKNOWN') AS postal_code,
        province,
        city_regency,
        district,
        subdistrict_village
    FROM silver.clean_scraped_data
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY COALESCE(NULLIF(TRIM(postal_code), ''), 'UNKNOWN') 
        ORDER BY cleaned_at DESC
    ) = 1
) AS s

ON t.postal_code = s.postal_code
WHEN MATCHED THEN 
    UPDATE SET *
WHEN NOT MATCHED THEN 
    INSERT *;

MERGE INTO gold.dim_location AS t
USING (
    SELECT 
        location_id,
        location_name,
        location_tags,
        address,
        latitude,
        longitude,
        location_url,
        website,
        phone_number,
        operating_hours,
        CURRENT_TIMESTAMP() AS updated_at
    FROM silver.clean_scraped_data
    WHERE location_id IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY location_id 
        ORDER BY updated_at DESC
    ) = 1
) AS s
ON t.location_id = s.location_id
WHEN MATCHED THEN 
    UPDATE SET *
WHEN NOT MATCHED THEN 
    INSERT *;


MERGE INTO gold.fact_location_metrics AS t
USING (
    SELECT 
        location_id,
        COALESCE(NULLIF(TRIM(postal_code), ''), 'UNKNOWN') AS postal_code,
        rating,
        reviews,
        scraped_at,
        cleaned_at
    FROM silver.clean_scraped_data
    WHERE location_id IS NOT NULL 
      AND scraped_at IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY location_id, scraped_at
        ORDER BY scraped_at DESC
    ) = 1
) AS s
ON t.location_id = s.location_id 
   AND t.scraped_at = s.scraped_at
WHEN MATCHED THEN 
    UPDATE SET *
WHEN NOT MATCHED THEN 
    INSERT *;